# Task 2: Exploratory Data Analysis on House Pricing Dataset

Dataset: `House_pricing.csv` (Ames Housing dataset — 1460 rows, 81 columns)

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('C:/Users/tasik/Python/datasets/House_pricing.csv')
df.shape

## 1. Exploratory Data Analysis

### 1.1 First and last few rows

In [ ]:
print(df.head())

In [ ]:
print(df.tail())

### 1.2 Dataset info — data types, memory footprint, missing values

In [ ]:
df.info(memory_usage='deep')

### 1.3 Summary statistics of numerical columns

In [ ]:
df.describe()

### 1.4 Count occurrences of each category in a label column
- #### Using `MSZoning` (the general zoning classification) as the label column.

In [ ]:
group = df.groupby(['MSZoning', 'Utilities'])['MSZoning'].count()
group

In [ ]:
valueCount = df[['MSZoning','Utilities']].value_counts()
valueCount

## 2. Data Selection

### 2.1 Select subset by specific columns

In [ ]:
cols_subset = df[['Id', 'Neighborhood', 'LotArea', 'YearBuilt', 'SalePrice']]
cols_subset.head()

### 2.2 Select subset by specific columns AND rows
- #### Using `.loc[]` to pick rows 0–9 and a set of columns.

In [ ]:
rows_cols_subset = df.loc[0:9, ['Neighborhood', 'OverallQual', 'SalePrice']]
rows_cols_subset

You can also filter rows by a condition, e.g. houses built after 2000:

In [ ]:
recent_houses = df.loc[df['YearBuilt'] > 2000, ['Neighborhood', 'YearBuilt', 'SalePrice']]
recent_houses.head()

## 3. Data Transformation

### 3.1 Apply a custom function to a specific column\n\nConvert `SalePrice` into price in thousands of dollars.

In [ ]:
def to_thousands(price):
    return round(price / 1000, 1)

#df['SalePrice_K'] = df['SalePrice'].apply((lambda x:x /1000))
#df['SalePrice_k']  = list(map((lambda x:x/1000), df['SalePrice']))
df['SalePrice_k'] = df['SalePrice'].map(lambda x:x/1000)
df[['SalePrice', 'SalePrice_k']].head()


### 3.2 Map values of a column using a dictionary
- #### Map `Street` (Grvl/Pave) to more descriptive labels.

In [ ]:
street_map = {'Grvl': 'Gravel', 'Pave': 'Paved'}
df['Street_full'] = df['Street'].map(street_map)
df[['Street', 'Street_full']].head()

## 4. Sorting

### Sort the dataset by a specific column in non-ascending (descending) order
- #### Sorting by `SalePrice` descending — most expensive houses first.

In [ ]:
df_sorted = df.sort_values(by='SalePrice', ascending=False)
df_sorted[['Id', 'Neighborhood', 'SalePrice']].head(10)

Sorting by multiple columns (`Neighborhood`, then `SalePrice`), both descending:

In [ ]:
df_sorted_multi = df.sort_values(by=['Neighborhood', 'SalePrice'], ascending=[False, False])
df_sorted_multi[['Neighborhood', 'SalePrice']].head(10)
df

## 5. Duplicate Handling

### 5.1 Check for duplicated rows in the dataset

In [ ]:
num_duplicate_rows = df.duplicated().sum()
print(f'Number of fully duplicated rows: {num_duplicate_rows}')

### 5.2 Check for duplicate entries in specific column(s)
- #### E.g. check if any houses share the same `Neighborhood` + `YearBuilt` + `LotArea` combination.

In [ ]:
dup_subset = df.duplicated(subset=['Neighborhood', 'YearBuilt', 'LotArea']).sum()
print(f'Duplicate entries based on Neighborhood, YearBuilt, LotArea: {dup_subset}')

### 5.3 Drop duplicate rows (if any)

In [ ]:
df_no_duplicates = df.drop_duplicates()
print('Shape before:', df.shape)
print('Shape after dropping duplicates:', df_no_duplicates.shape)

## 6. Missing Value Handling

### 6.1 Check if the dataset contains any missing values

In [ ]:
print('Any missing values in dataset:', df.isnull().values.any())
print()
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print('Columns with missing values:')
missing_counts

### 6.2 Fill missing values with the next valid observation
- #### `bfill` (backward fill) propagates the *next* valid value backward to fill NaNs.

In [ ]:
df_filled = df.bfill()
print('Missing values before fill:', df.isnull().sum().sum())
print('Missing values after bfill:', df_filled.isnull().sum().sum())

Note: if the *last* row(s) of a column are also NaN, `bfill` alone can't fill them (there's no valid observation after them). We can check and optionally follow up with `ffill` for any leftovers:

In [ ]:
remaining_na = df_filled.isnull().sum()
remaining_na = remaining_na[remaining_na > 0]
print(remaining_na)